[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/pinecone-io/examples/blob/master/learn/generation/openai/openai-ml-qa/01-making-queries.ipynb) [![Open nbviewer](https://raw.githubusercontent.com/pinecone-io/examples/master/assets/nbviewer-shield.svg)](https://nbviewer.org/github/pinecone-io/examples/blob/master/learn/generation/openai/openai-ml-qa/01-making-queries.ipynb)

# Making Queries

In this notebook we will learn how to query relevant contexts to our queries from Pinecone, and pass these to a generative OpenAI model to generate an answer backed by real data sources. Required installs for this notebook are:

In [ ]:
!pip install -qU openai==1.86.0 pinecone~=8.0

import os

from openai import OpenAI
from pinecone import Pinecone

---

🚨 _Note: the above `pip install` is formatted for Jupyter notebooks. If running elsewhere you may need to drop the `!`._

---

## Initializing Everything

We will start by initializing everything we will be using. Those components are:

* Pinecone vector DB for retrieval (we must also connect to the previously build `openai-ml-qa` index)

* OpenAI `text-embedding-ada-002` embedding model for embedding queries

* OpenAI `text-davinci-003` generation model for generating answers

We first initialize the vector DB. For that we need our [free Pinecone API key](https://app.pinecone.io).

In [ ]:
# initialize connection to pinecone (get API key at app.pinecone.io)
api_key = os.environ.get("PINECONE_API_KEY") or "PINECONE_API_KEY"

# configure client
pc = Pinecone(api_key=api_key)

Now connect to our existing index:

In [3]:
index_name = "openai-ml-qa"

In [4]:
# connect to index
index = pc.Index(index_name)
# view index stats
index.describe_index_stats()

{'dimension': 1536,
 'index_fullness': 0.0,
 'namespaces': {'': {'vector_count': 5458}},
 'total_vector_count': 5458}

Now initialize the OpenAI models (or _"engines"_), for this we need an [OpenAI API key](https://platform.openai.com/).

In [ ]:
# get API key from top-right dropdown on OpenAI website
openai_api_key = os.getenv("OPENAI_API_KEY") or "OPENAI_API_KEY"

# initialize OpenAI client
openai_client = OpenAI(api_key=openai_api_key)

We will use the embedding model `text-embedding-ada-002` like so:

In [ ]:
embed_model = "text-embedding-ada-002"

query = "What are the differences between PyTorch and TensorFlow?"

res = openai_client.embeddings.create(input=[query], model=embed_model)

And use the returned query vector to query Pinecone like so:

In [ ]:
xq = res.data[0].embedding

res = index.query(vector=xq, top_k=3, include_metadata=True)
res

We have some relevant contexts there, and some irrelevant. Now we rely on the generative model `text-davinci-003` to generate our answer.

In [ ]:
limit = 3750

contexts = [x["metadata"]["context"] for x in res["matches"]]

# build our prompt with the retrieved contexts included
prompt_start = "Answer the question based on the context below.\n\n" + "Context:\n"
prompt_end = f"\n\nQuestion: {query}\nAnswer:"
# append contexts until hitting limit
for i in range(1, len(contexts)):
    if len("\n\n---\n\n".join(contexts[:i])) >= limit:
        prompt = prompt_start + "\n\n---\n\n".join(contexts[: i - 1]) + prompt_end
        break
    elif i == len(contexts) - 1:
        prompt = prompt_start + "\n\n---\n\n".join(contexts) + prompt_end

# now query gpt-3.5-turbo using chat completions
res = openai_client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[{"role": "user", "content": prompt}],
    temperature=0,
    max_tokens=400,
    top_p=1,
    frequency_penalty=0,
    presence_penalty=0,
    stop=None,
)

We check the generated response like so:

In [ ]:
res.choices[0].message.content.strip()

What we get here essentially an extract from the top result, we can ask for more information by modifying the prompt.

In [ ]:
query = "What are the differences between PyTorch and TensorFlow?"

# create query vector
res = openai_client.embeddings.create(input=[query], model=embed_model)
xq = res.data[0].embedding

# get relevant contexts
res = index.query(vector=xq, top_k=10, include_metadata=True)
contexts = [x["metadata"]["context"] for x in res["matches"]]

# build our prompt with the retrieved contexts included
prompt_start = (
    "Give an exhaustive summary and answer based on the question using the contexts below.\n\n"
    + "Context:\n"
    + "\n\n---\n\n".join(contexts)
    + "\n\n"
    + f"Question: {query}\n"
    + "Answer:"
)
prompt_end = f"\n\nQuestion: {query}\nAnswer:"
# append contexts until hitting limit
for i in range(1, len(contexts)):
    if len("\n\n---\n\n".join(contexts[:i])) >= limit:
        prompt = prompt_start + "\n\n---\n\n".join(contexts[: i - 1]) + prompt_end
    elif i == len(contexts):
        prompt = prompt_start + "\n\n---\n\n".join(contexts) + prompt_end

# now query gpt-3.5-turbo using chat completions
res = openai_client.chat.completions.create(
    model="gpt-3.5-turbo",
    messages=[{"role": "user", "content": prompt}],
    temperature=0,
    max_tokens=400,
    top_p=1,
    frequency_penalty=0,
    presence_penalty=0,
    stop=None,
)
res.choices[0].message.content.strip()

The advantage of Tensorflow.js could have been framed better and the fact that PyTorch has no equivalent explicitly stated. However, the answer is good and gives a nice summary and answer to our question — using information pulled from multiple sources.

Once you're finished with the index we delete it to save resources:

In [11]:
pc.delete_index(index_name)

---